In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install huggingface_hub
from huggingface_hub import login
import shutil

from huggingface_hub import snapshot_download

HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    login(token=HF_TOKEN)
path = snapshot_download(
    repo_id="ibrahimhamamci/DENTEX",
    repo_type="dataset"
)

print(path)

print("DENTEX ready")

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

/root/.cache/huggingface/hub/datasets--ibrahimhamamci--DENTEX/snapshots/7b27ccc8e342dcb774f69adc6ca5e6c09fefce93
DENTEX ready


In [3]:
import zipfile
import os

base_path = "/root/.cache/huggingface/hub/datasets--ibrahimhamamci--DENTEX/snapshots/7b27ccc8e342dcb774f69adc6ca5e6c09fefce93/DENTEX"

for file in ["training_data.zip", "validation_data.zip", "test_data.zip"]:
    zip_path = os.path.join(base_path, file)
    extract_path = os.path.join(base_path, file.replace(".zip", ""))

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

    print(f"Extracted {file}")

Extracted training_data.zip
Extracted validation_data.zip
Extracted test_data.zip


In [5]:
!pip install -q ultralytics qwen-vl-utils accelerate
!pip install -q -U transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 19.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 49.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 79.1 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 60.4 MB/s eta 0:00:00:00:01


In [7]:
import os, gc, json, time, tempfile
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as tv_models
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO
from transformers import AutoProcessor


YOLO_DISEASE = "/kaggle/input/models/boulesnane/yolo-disease/other/default/1/Yolo_disease.pt"
YOLO_TEETH   = "/kaggle/input/models/boulesnane/bestyoloteethfliplr0/other/default/1/best.pt"
UNET_SEG     = "/kaggle/input/models/boulesnane/unet-resnet34/pytorch/default/1/U-net_ResNet34.pth"

disease_model = YOLO(YOLO_DISEASE)
teeth_model   = YOLO(YOLO_TEETH)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [9]:
# ── START: prepare DENTEX fold-5 val ground truth + helpers for the threshold ablation ──
import os, json, math
import numpy as np
from collections import Counter
from sklearn.model_selection import StratifiedKFold
from ultralytics import YOLO

FOLD, SEED = 5, 42
DX = ("/root/.cache/huggingface/hub/datasets--ibrahimhamamci--DENTEX/snapshots/"
      "7b27ccc8e342dcb774f69adc6ca5e6c09fefce93/DENTEX")
first = lambda *c: next(p for p in c if os.path.exists(p))
TR_DIR = first(f"{DX}/training_data/quadrant-enumeration-disease",
               f"{DX}/training_data/training_data/quadrant-enumeration-disease")
VA_DIR = first(f"{DX}/validation_data/quadrant_enumeration_disease",
               f"{DX}/validation_data/validation_data/quadrant_enumeration_disease")

DISEASE_NAMES = {0: "Impacted", 1: "Caries", 2: "Periapical Lesion", 3: "Deep Caries"}
FDI_LIST      = [q * 10 + t for q in range(1, 5) for t in range(1, 9)]

# same split as the disease-model CV (train images first, then val; dominant disease; StratifiedKFold)
dataset = {}
for src, js, img_dir in [("train", f"{TR_DIR}/train_quadrant_enumeration_disease.json", f"{TR_DIR}/xrays"),
                         ("val",   f"{DX}/validation_triple.json",                     f"{VA_DIR}/xrays")]:
    d = json.load(open(js))
    for im in d["images"]:
        dataset[(src, im["id"])] = {"info": im, "source": img_dir, "annotations": []}
    for a in d["annotations"]:
        if (src, a["image_id"]) in dataset:
            dataset[(src, a["image_id"])]["annotations"].append(a)
keys = list(dataset)
strat = [Counter(a["category_id_3"] for a in dataset[k]["annotations"]).most_common(1)[0][0]
         if dataset[k]["annotations"] else -1 for k in keys]
val_idx = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(keys, strat))[FOLD - 1][1]

reconstructed_d = []
for i in val_idx:
    s = dataset[keys[i]]
    reconstructed_d.append({
        "file_name":  f"{keys[i][0]}/{s['info']['file_name']}",          # unique (train_12 vs val_12)
        "image_path": os.path.join(s["source"], s["info"]["file_name"]),
        "findings":   [{"fdi": (a["category_id_1"] + 1) * 10 + a["category_id_2"] + 1,
                        "disease": DISEASE_NAMES[a["category_id_3"]]} for a in s["annotations"]],
    })
print(f"DENTEX fold-{FOLD} val: {len(reconstructed_d)} images, "
      f"{sum(len(r['findings']) for r in reconstructed_d)} findings")

# helpers used by the ablation (defined here in case the pipeline cells were not run)
def compute_fusion_score(d, t):                     # Eq. (3) on normalized xywh boxes
    ax1, ay1, ax2, ay2 = d[0]-d[2]/2, d[1]-d[3]/2, d[0]+d[2]/2, d[1]+d[3]/2
    bx1, by1, bx2, by2 = t[0]-t[2]/2, t[1]-t[3]/2, t[0]+t[2]/2, t[1]+t[3]/2
    inter = max(0, min(ax2, bx2) - max(ax1, bx1)) * max(0, min(ay2, by2) - max(ay1, by1))
    iou = inter / ((ax2-ax1)*(ay2-ay1) + (bx2-bx1)*(by2-by1) - inter + 1e-12)
    return 0.7 * iou + 0.3 * math.exp(-10 * math.hypot(d[0]-t[0], d[1]-t[1]))

# models (only loaded if not already in memory)
if "disease_model" not in globals():
    disease_model = YOLO("/kaggle/input/models/boulesnane/yolo-disease/other/default/1/Yolo_disease.pt")
if "teeth_model" not in globals():
    teeth_model = YOLO("/kaggle/input/models/boulesnane/bestyoloteethfliplr0/other/default/1/best.pt")
print("disease model trained on:", disease_model.ckpt["train_args"]["data"])   # must end in fold_5/...
if "OUTPUT_DIR" not in globals():
    OUTPUT_DIR = "/kaggle/working"
# ── END ──

DENTEX fold-5 val: 151 images, 772 findings
disease model trained on: /kaggle/working/dentex_yolo_cv/fold_5/dataset.yaml


In [11]:
# ── START: disease YOLO alone — box-level precision / recall per confidence threshold ──
import numpy as np, pandas as pd, json
from collections import defaultdict

THRESHOLDS = [0.15, 0.25, 0.35, 0.45, 0.50, 0.60, 0.70, 0.80]
IOU_THR    = 0.50
NAME2ID    = {v: k for k, v in DISEASE_NAMES.items()}

# ground-truth disease boxes (xyxy, pixels) from the same fold-5 val images
gt_boxes = {}
for key_i in val_idx:
    s = dataset[keys[key_i]]
    name = f"{keys[key_i][0]}/{s['info']['file_name']}"
    gt_boxes[name] = [([a["bbox"][0], a["bbox"][1], a["bbox"][0] + a["bbox"][2], a["bbox"][1] + a["bbox"][3]],
                       a["category_id_3"]) for a in s["annotations"]]

def iou(a, b):
    iw = max(0, min(a[2], b[2]) - max(a[0], b[0])); ih = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = iw * ih
    return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter + 1e-9)

# predictions once at conf 0.01
PRED = {}
for item in reconstructed_d:
    r = disease_model.predict(item["image_path"], conf=0.01, imgsz=1024, verbose=False)[0]
    PRED[item["file_name"]] = list(zip(r.boxes.xyxy.tolist(), r.boxes.cls.int().tolist(), r.boxes.conf.tolist()))

rows = []
for t in THRESHOLDS:
    cls = defaultdict(lambda: [0, 0, 0])                  # tp, fp, fn per class
    for name, preds in PRED.items():
        gts = gt_boxes[name]; used = set()
        for b, c, cf in sorted([p for p in preds if p[2] >= t], key=lambda p: -p[2]):
            best, bi = 0, None
            for j, (g, gc) in enumerate(gts):
                if j in used or gc != c: continue
                v = iou(b, g)
                if v > best: best, bi = v, j
            if best >= IOU_THR:
                used.add(bi); cls[c][0] += 1
            else:
                cls[c][1] += 1
        for j, (g, gc) in enumerate(gts):
            if j not in used: cls[gc][2] += 1
    tp, fp, fn = (sum(v[i] for v in cls.values()) for i in range(3))
    P, R = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    row = {"conf": t, "boxes_per_img": (tp + fp) / len(PRED), "precision": P, "recall": R,
           "F1": 2 * P * R / max(P + R, 1e-9)}
    for c, (a, b, d) in sorted(cls.items()):
        row[f"{DISEASE_NAMES[c]} P"] = a / max(a + b, 1); row[f"{DISEASE_NAMES[c]} R"] = a / max(a + d, 1)
    rows.append(row)

TABD = pd.DataFrame(rows).set_index("conf")
pd.set_option("display.width", 250)
print(f"Disease YOLO alone (IoU >= {IOU_THR}, same class) — DENTEX fold-{FOLD} val, {len(PRED)} images\n")
print(TABD.round(3).to_string())
print(f"\nBest F1 at conf = {TABD['F1'].idxmax()}")
TABD.to_csv(f"{OUTPUT_DIR}/threshold_ablation_disease_yolo_only.csv")
# ── END ──

Disease YOLO alone (IoU >= 0.5, same class) — DENTEX fold-5 val, 151 images

      boxes_per_img  precision  recall     F1  Impacted P  Impacted R  Caries P  Caries R  Periapical Lesion P  Periapical Lesion R  Deep Caries P  Deep Caries R
conf                                                                                                                                                             
0.15          7.179      0.476   0.668  0.556       0.865       0.910     0.420     0.631                0.289                0.629          0.472          0.562
0.25          5.483      0.559   0.600  0.579       0.871       0.910     0.501     0.529                0.368                0.600          0.533          0.537
0.35          4.331      0.636   0.539  0.583       0.884       0.910     0.589     0.438                0.444                0.571          0.558          0.521
0.45          3.576      0.669   0.468  0.550       0.898       0.858     0.606     0.349                0.556   

In [12]:
# ── START: tooth YOLO alone — per confidence threshold (DENTEX enumeration, fold-4 val) ──
import os, json, numpy as np, pandas as pd
from sklearn.model_selection import KFold
from collections import defaultdict

TOOTH_FOLD = 4
THRESHOLDS = [0.10, 0.15, 0.25, 0.35, 0.45, 0.50, 0.60, 0.70, 0.80]
IOU_THR    = 0.50
AGNOSTIC   = True                       # pipeline setting
DX = ("/root/.cache/huggingface/hub/datasets--ibrahimhamamci--DENTEX/snapshots/"
      "7b27ccc8e342dcb774f69adc6ca5e6c09fefce93/DENTEX")
ENUM_DIR = next(p for p in [f"{DX}/training_data/quadrant_enumeration",
                            f"{DX}/training_data/training_data/quadrant_enumeration"] if os.path.exists(p))
coco = json.load(open(f"{ENUM_DIR}/train_quadrant_enumeration.json"))
IMG_DIR = f"{ENUM_DIR}/xrays"
FDI_LIST = [q * 10 + t for q in range(1, 5) for t in range(1, 9)]
ALL_TEETH = set(FDI_LIST)

# same split as the tooth-model CV
by_img = defaultdict(list)
for a in coco["annotations"]:
    by_img[a["image_id"]].append(a)
info = {im["id"]: im for im in coco["images"]}
annotated = sorted(info[i]["file_name"] for i in by_img if os.path.exists(os.path.join(IMG_DIR, info[i]["file_name"])))
name2id = {info[i]["file_name"]: i for i in by_img}
val_names = [annotated[i] for i in list(KFold(5, shuffle=True, random_state=42).split(annotated))[TOOTH_FOLD - 1][1]]

GT_T = {}                                   # name -> [(xyxy, fdi)]
for fn in val_names:
    GT_T[fn] = [([a["bbox"][0], a["bbox"][1], a["bbox"][0] + a["bbox"][2], a["bbox"][1] + a["bbox"][3]],
                 (a["category_id_1"] + 1) * 10 + a["category_id_2"] + 1) for a in by_img[name2id[fn]]]

def iou(a, b):
    iw = max(0, min(a[2], b[2]) - max(a[0], b[0])); ih = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = iw * ih
    return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter + 1e-9)

# predictions once at a low threshold
PRED_T = {}
for fn in val_names:
    r = teeth_model.predict(os.path.join(IMG_DIR, fn), conf=0.01, agnostic_nms=AGNOSTIC, verbose=False)[0]
    PRED_T[fn] = [(b, FDI_LIST[int(c)], cf) for b, c, cf in
                  zip(r.boxes.xyxy.tolist(), r.boxes.cls.tolist(), r.boxes.conf.tolist())]

rows = []
for t in THRESHOLDS:
    tp = fp = fn_ = 0; m_tp = m_fp = m_fn = 0
    for name in val_names:
        preds = sorted([p for p in PRED_T[name] if p[2] >= t], key=lambda p: -p[2])
        gts, used = GT_T[name], set()
        for b, f, cf in preds:                                     # box level: same FDI + IoU
            best, bi = 0, None
            for j, (g, gf) in enumerate(gts):
                if j in used or gf != f: continue
                v = iou(b, g)
                if v > best: best, bi = v, j
            if best >= IOU_THR: used.add(bi); tp += 1
            else: fp += 1
        fn_ += len(gts) - len(used)
        # missing-teeth list, exactly as given to the VLM (best box per FDI)
        pred_present = {f for _, f, _ in preds}
        gt_missing   = ALL_TEETH - {gf for _, gf in gts}
        pr_missing   = ALL_TEETH - pred_present
        m_tp += len(pr_missing & gt_missing); m_fp += len(pr_missing - gt_missing); m_fn += len(gt_missing - pr_missing)
    P, R = tp / max(tp + fp, 1), tp / max(tp + fn_, 1)
    mP, mR = m_tp / max(m_tp + m_fp, 1), m_tp / max(m_tp + m_fn, 1)
    rows.append({"conf": t, "teeth_per_img": (tp + fp) / len(val_names),
                 "precision": P, "recall": R, "F1": 2 * P * R / max(P + R, 1e-9),
                 "missing P": mP, "missing R": mR, "missing F1": 2 * mP * mR / max(mP + mR, 1e-9),
                 "missing listed/img": (m_tp + m_fp) / len(val_names)})

TABT = pd.DataFrame(rows).set_index("conf")
print(f"Tooth YOLO alone — DENTEX enumeration fold-{TOOTH_FOLD} val ({len(val_names)} images), "
      f"agnostic NMS={AGNOSTIC}, IoU>={IOU_THR}, same FDI\n")
print(TABT.round(3).to_string())
print(f"\nBest box F1 at conf = {TABT['F1'].idxmax()} | best missing-teeth F1 at conf = {TABT['missing F1'].idxmax()}")
print(f"Pipeline setting 0.25: box P {TABT.loc[0.25,'precision']:.3f} R {TABT.loc[0.25,'recall']:.3f} | "
      f"missing P {TABT.loc[0.25,'missing P']:.3f} R {TABT.loc[0.25,'missing R']:.3f}")
TABT.to_csv(f"{OUTPUT_DIR}/threshold_ablation_tooth_yolo_only.csv")
# ── END ──

Tooth YOLO alone — DENTEX enumeration fold-4 val (127 images), agnostic NMS=True, IoU>=0.5, same FDI

      teeth_per_img  precision  recall     F1  missing P  missing R  missing F1  missing listed/img
conf                                                                                               
0.10         28.898      0.913   0.933  0.922      0.817      0.789       0.803               3.614
0.15         28.693      0.919   0.932  0.926      0.815      0.796       0.805               3.654
0.25         28.457      0.924   0.930  0.927      0.806      0.813       0.809               3.772
0.35         28.346      0.926   0.929  0.928      0.798      0.817       0.807               3.827
0.45         28.189      0.929   0.926  0.928      0.780      0.821       0.800               3.937
0.50         28.039      0.931   0.923  0.927      0.759      0.823       0.790               4.055
0.60         27.472      0.936   0.909  0.923      0.690      0.844       0.759               4.57